# Google Translate Dataset to English

This notebook reads an Excel file and translates the comment columns to English using **Google Translate** (via the free `deep-translator` library).

The dataset is mostly **Malay** with a lot of **English** and one **Chinese** comment, so the language handling is tuned for that:
comments already in English are left as-is, Chinese is detected and translated, and everything else is treated as Malay.

No API key needed. Works on CPU. Internet connection required.

## Objectives
### 1. To develop a sentiment analysis system for Malaysian festival social media text, specifically the Rain Rave Music Festival, to help the government and event organisers gauge public opinion and guide future event planning.



## 1. Install Libraries

In [9]:
%pip install -q deep-translator langdetect pandas openpyxl tqdm

Note: you may need to restart the kernel to use updated packages.


## 2. Set Input and Output Path

In [10]:
from pathlib import Path

# Change these two paths when you want to use another file.
INPUT_FILE = r"D:\NLP Project\V2_NLP Project_Dataset.xlsx"
OUTPUT_FILE = r"D:\NLP Project\V2_NLP Project_Dataset_English.csv"

# Columns that should be translated.
TEXT_COLUMNS = [
    "Positive Comments",
    "Neutral Comments",
    "Negative Comments",
]

TARGET_LANG = "en"

# The dataset is mostly Malay + English with one Chinese comment.
# Anything that is not English and not Chinese is treated as Malay.
DEFAULT_SOURCE_LANG = "ms"   # Malay (deep-translator / Google code)

# Google Translate caps each request at ~5000 chars.
MAX_CHARS = 4500

## 3. Load Excel Dataset

In [11]:
import pandas as pd

df = pd.read_excel(INPUT_FILE)

missing_columns = [col for col in TEXT_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"These columns are missing: {missing_columns}")

# The bottom of the sheet contains reference links (TikTok / Facebook /
# Instagram / Reddit) that are NOT comments. Only the rows with a valid
# 'No.' are real comments, so keep just those (the 130 numbered rows).
df = df[df["No."].notna()].reset_index(drop=True)

translated_df = df.copy()

print("Dataset shape (comments only):", df.shape)
print("Non-null counts per column:")
print(df[TEXT_COLUMNS].count())
display(df.head())

Dataset shape (comments only): (130, 4)
Non-null counts per column:
Positive Comments    130
Neutral Comments     130
Negative Comments    130
dtype: int64


,No.,Positive Comments,Neutral Comments,Negative Comments
0,1.0,xsabar nak tengok wukong😍❤️,masuk free ke,nk sgt mandi hujan kn...YA ALLAH turunkan lah ...
1,2.0,"Dolla, joe flizow&mimi fly what a date perform...",konsert tarikh bila ?,Dengan lafaz Bismillahirahmanirahim. Aku memoh...
2,3.0,yes awesome. who is going jom,How to get this ticket ?,Whoever approved this location next to Pavilio...
3,4.0,omg a reason for me to finally go back kl 😝,festival tanah ada ke,"Just so noisy at there, idiot only go"
4,5.0,I came across plenty of positive feedback from...,i thought is 1-3may?,Membazir air betul lah


## 4. Set Up Google Translator

In [12]:
from deep_translator import GoogleTranslator

# We pick the source language per comment (see helper functions below),
# so we create translators on demand and cache them here.
translators = {}

def get_translator(source_lang):
    if source_lang not in translators:
        translators[source_lang] = GoogleTranslator(source=source_lang, target=TARGET_LANG)
    return translators[source_lang]

# Quick test
print(get_translator("ms").translate("Selamat pagi, apa khabar?"))
print(get_translator("zh-CN").translate("今天天气很好"))

Good morning, how are you?
The weather is very good today


## 5. Helper Functions

In [13]:
import re
import time
from langdetect import detect, DetectorFactory

# Make langdetect deterministic (same input -> same result every run).
DetectorFactory.seed = 0

# Matches Chinese (CJK) characters.
chinese_char = re.compile(r"[\u4e00-\u9fff]")

# A few common Malay words. If a comment contains any of these, we treat it
# as Malay even if langdetect is unsure (helps with code-switched text).
MALAY_HINT_WORDS = set("""
aku saya kami kita korang korg dia di ke dari dgn dengan yg yang lah pun ni tu
tak nak nk dah je betul sangat sgt kalau boleh ada ade dekat masuk mandi tidak
hujan air pesta kerajaan anak cucu rumah rasa seronok untuk dalam ini itu dan
jom mari lah kan eh wei weh bro sis macam mcm tau tahu sini situ buat gi pergi
""".split())

def is_empty(value):
    return pd.isna(value) or not str(value).strip()

def has_malay_hint(text):
    words = re.findall(r"[A-Za-z']+", text.lower())
    return any(word in MALAY_HINT_WORDS for word in words)

def choose_source_lang(text):
    """Decide the source language for one comment.
    Returns a Google language code, or None if the text is already English.
    """
    text = str(text).strip()

    # 1. Any Chinese characters -> Chinese.
    if chinese_char.search(text):
        return "zh-CN"

    # 2. Obvious Malay words -> Malay (handles Malay/English code-switching).
    if has_malay_hint(text):
        return "ms"

    # 3. Otherwise ask langdetect.
    try:
        detected = detect(text)
    except Exception:
        detected = "unknown"

    # Already English -> no translation needed.
    if detected == "en":
        return None

    # Indonesian is very close to Malay; Google handles it well as Malay.
    if detected in ("ms", "id"):
        return "ms"

    if detected.startswith("zh"):
        return "zh-CN"

    # Fallback: treat anything else as Malay (our dominant non-English language).
    return DEFAULT_SOURCE_LANG

# Cache so identical comments are only sent once (saves time + requests).
translation_cache = {}

def translate_one(value, retries=3):
    if is_empty(value):
        return value

    text = str(value).strip()

    source_lang = choose_source_lang(text)

    # Already English -> keep as-is.
    if source_lang is None:
        return text

    cache_key = (source_lang, text)
    if cache_key in translation_cache:
        return translation_cache[cache_key]

    snippet = text[:MAX_CHARS]
    translator = get_translator(source_lang)

    for attempt in range(retries):
        try:
            result = translator.translate(snippet)
            if not result or not str(result).strip():
                result = text
            translation_cache[cache_key] = result
            return result
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
            else:
                print(f"  [warn] failed to translate ({source_lang}) after {retries} tries: {e}")
                translation_cache[cache_key] = text
                return text

## 6. Translate the Dataset

In [14]:
from tqdm.auto import tqdm

tqdm.pandas()

for column in TEXT_COLUMNS:
    print(f"Translating column: {column}")
    translated_df[column] = translated_df[column].progress_apply(translate_one)

display(translated_df.head())

Translating column: Positive Comments


100%|██████████| 130/130 [00:02<00:00, 51.32it/s]


Translating column: Neutral Comments


100%|██████████| 130/130 [00:02<00:00, 43.37it/s]


Translating column: Negative Comments


100%|██████████| 130/130 [00:05<00:00, 25.53it/s]


,No.,Positive Comments,Neutral Comments,Negative Comments
0,1.0,I can't wait to see Wukong😍❤️,enter for free,I really want to take a shower in the rain... ...
1,2.0,"Dolla, joe flizow&mimi fly what a date perform...",When is the concert date?,With the pronunciation of Bismillahirahmanirah...
2,3.0,yes awesome. who is going let's go,How to get this ticket ?,Whoever approved this location next to Pavilio...
3,4.0,omg a reason for me to finally go back kl 😝,the land festival is there,"Just so noisy at there, idiot only go"
4,5.0,I came across plenty of positive feedback from...,i thought is 1-3may?,It's a waste of water


## 7. Save English-Only Output

In [15]:
output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)

# Write according to the file extension.
if output_path.suffix.lower() == ".csv":
    # utf-8-sig so Excel shows any non-ASCII characters correctly.
    translated_df.to_csv(output_path, index=False, encoding="utf-8-sig")
else:
    translated_df.to_excel(output_path, index=False)

print("Saved English file to:")
print(output_path.resolve(), "| exists:", output_path.exists())

Saved English file to:
D:\NLP Project\V2_NLP Project_Dataset_English.csv | exists: True
